In [2]:
%cd ~/Desktop/線上課程教材/pythan基礎觀念/爬蟲/好球帶研究室
%load_ext autoreload
%autoreload 2

/Users/leochen/Desktop/線上課程教材/pythan基礎觀念/爬蟲/好球帶研究室


In [68]:
%%writefile src/fetch.py

import requests
import json
from bs4 import BeautifulSoup
import time
import pandas as pd
from pathlib import Path

SEASON='CPBL-2026-oB'
def get_json(path, retries=3, pause=1.0):
    for attempt in range(1,retries+1):
        try:
            session=requests.Session()
            ua='Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/150.0.0.0 Safari/537.36'
            
            res=session.get(path,headers={'User-agent':ua})
            res.status_code
            time.sleep(pause)
            return res.json()
        except (requests.RequestException, ValueError) as e:
            print(f'第{attempt}次失敗 {path} 原因：{e}')
            time.sleep(pause* attempt * 2)
    raise RuntimeError(f'抓不到資料{path}')


def cath_json(path, catch_name, refresh=True):
    path_way=Path(__file__).resolve().parent.parent/"data"/"raw"
    cath=path_way/ catch_name
    cath.parent.mkdir(parents=True,exist_ok=True)
    if cath.exists() and not refresh:
        return json.loads(cath.read_text(encoding="utf-8"))
    data=get_json(path)
    cath.write_text(json.dumps(data, ensure_ascii=False), encoding="utf-8")
    return data



Overwriting src/fetch.py


In [69]:
%%writefile -a src/fetch.py
def team_info(season=SEASON,refresh=True):
    data=cath_json(f'https://www.rebas.tw/api/seasons/{season}/leaders?type=team&section=standard&pa=undefined',f'{season}_team.json',refresh)['data']
    row=[]
    for i in data:
        o=i['team']
        row.append({
            "team_id": o["uniqid"], "team": o["name"], "abbr": o["abbr"],
                     "color": o["hex_color"].split(";")[0]
        })
    return pd.DataFrame(row)

Appending to src/fetch.py


In [70]:
%%writefile -a src/fetch.py
def team_standing(season=SEASON,refresh=True):
    data=cath_json(f'https://www.rebas.tw/api/seasons/{season}/leaders?type=team&section=standard&pa=undefined',f'{season}_standing.json',refresh)['data']
    row=[]
    for i in data:
        row.append({"rank": i["order"], "team": i["team"]["name"], "abbr": i["team"]["abbr"],
                     "W": i["wins"], "L": i["loses"], "T": i["draws"], "PCT": i["PCT"],
                     "GB": i["GB"], "DIFF": i["DIFF"]})
    return pd.DataFrame(row).set_index('rank')

Appending to src/fetch.py


In [75]:
from src import fetch
print(fetch.team_standing())

             team abbr   W   L  T    PCT    GB  DIFF
rank                                                
1             味全龍    龍  67  53  0  0.558   0.0    70
2     統一7-ELEVEn獅    獅  64  55  1  0.538   2.5    48
3            富邦悍將    悍  59  61  0  0.492   8.0   -24
4            樂天桃猿    猿  57  61  2  0.483   9.0   -53
5            中信兄弟    象  56  62  2  0.475  10.0     9
6            台鋼雄鷹    鷹  54  65  1  0.454  12.5   -50


In [76]:
!git add .
!git commit -m "Day 3: 爬蟲共用工具、球隊與戰績"

[main e6d5d3a] Day 3: 爬蟲共用工具、球隊與戰績
 Committer: Leo Chen <leochen@LeodeMacBook-Air.local>
Your name and email address were configured automatically based
on your username and hostname. Please check that they are accurate.
You can suppress this message by setting them explicitly. Run the
following command and follow the instructions in your editor to edit
your configuration file:

    git config --global --edit

After doing this, you may fix the identity used for this commit with:

    git commit --amend --reset-author

 5 files changed, 383 insertions(+), 15 deletions(-)
 create mode 100644 .virtual_documents/D02.ipynb
 create mode 100644 D03.ipynb
 create mode 100644 src/fetch.py


In [79]:
%%writefile -a src/fetch.py

def fetch_games(season=SEASON, refresh=True):
    data=cath_json(f'https://www.rebas.tw/api/seasons/{season}/games',f'games_{season}.json',refresh)['data']
    row=[]
    for i in data:
        if i['info']['status']!='FINISHED':
            continue
        row.append({"game_id": i["uniqid"], "seq": i["seq"],
                     "date": i["info"]["scheduled_start_at"][:10],
                     "away": i["away"]["team"], "home": i["home"]["team"],
                     "away_runs": i["away"]["runs"], "home_runs": i["home"]["runs"],
                     "winner_side": i["info"]["winner_side"]})
    return pd.DataFrame(row)

Appending to src/fetch.py


In [81]:
from src import fetch
games=fetch.fetch_games()
print(games.shape)
home=games.groupby('home').agg(rs=('home_runs','sum'),ra=('away_runs','sum'))
away=games.groupby('away').agg(rs=('away_runs','sum'),ra=('home_runs','sum'))
runs=home.add(away,fill_value=0)
print(runs)

(360, 8)
              rs   ra
home                 
中信兄弟         509  500
台鋼雄鷹         456  506
味全龍          441  371
富邦悍將         434  458
樂天桃猿         390  443
統一7-ELEVEn獅  450  402


In [82]:
%%writefile src/config.py
"""全站設定：台灣大賽對戰組合與主場安排。"""
# 第一個是全年勝率最高、直接晉級的球隊；第二個是另一支晉級的球隊
TS_TEAMS = ("味全龍", "中信兄弟")
# 第一隊每一場是否為主場（勝率最高的球隊最多可有 5 場主場）
TS_HOME_PATTERN = [True, True, False, False, True, True, True]
TS_NEED = 4                     # 七戰四勝
HOME_ADVANTAGE = 0.023          # 主場優勢：中職 2023–2026 主場勝率 .523（1362 場），比 50% 多 2.3 個百分點
POSTSEASON_TITLE = "台灣大賽"    # 最後一天揭曉時，用來在野球革命的賽季列表中找到台灣大賽

Writing src/config.py


In [85]:
from src import fetch

test_data=fetch.cath_json(
'https://www.rebas.tw/api/seasons/CPBL-2026-oB/leaders?type=team&section=standard&pa=undefined','test_data_standard.json')['data']

df_test=pd.json_normalize(test_data)
print(df_test.columns.to_list)

<bound method IndexOpsMixin.tolist of Index(['DIFF', 'E', 'GB', 'M', 'PCT', 'STRK', 'draws', 'finished',
       'last_updated_at', 'loses', 'order', 'total', 'wins', 'team.abbr',
       'team.hex_color', 'team.name', 'team.season_team_uniqid',
       'team.uniqid'],
      dtype='object')>


In [87]:
%%writefile -a src/fetch.py

def fetch_player(kind,season=SEASON,refresh=True):
    data_all=[]
    for section in ['standard','advanced','new']:
        url=f'https://www.rebas.tw/api/seasons/{season}/leaders?type={kind}&section={section}&pa=undefined'
        data=cath_json(url,f'{season}_{kind}_{section}.json',refresh=refresh)['data']
        df=pd.json_normalize(data)
        df = df.rename(columns={"player.uniqid": "player_id", "player.name": "name",
                                "player.team_abbr": "abbr"})
        data_all.append(df)
    std,adv,new=data_all
    
    keep=['player_id']+[c for c in adv.columns if c not in std.columns]
    merge_1=std.merge(adv[keep],on='player_id',how='left')
    keep_1=['player_id']+[d for d in new.columns if d not in merge_1.columns]
    merge_2=merge_1.merge(new[keep_1],on='player_id',how='left')
    drop=[c for c in merge_2.columns if c.startswith('player')]+["reach_min"]
    return merge_2.drop(columns=[c for c in drop if c in merge_2.columns])


Appending to src/fetch.py


In [88]:
from src import fetch
b=fetch.fetch_player('batter')
c=fetch.fetch_player('pitcher')
print(b.shape)
print(c.shape)

(183, 41)
(181, 47)


In [95]:
q=b[b['PA']>372].nlargest(5,'wOBA')
print(q[['name','abbr','wOBA']])
a=c[c['IPOut']>120].nsmallest(5,'FIP')
print(a[['name','abbr','FIP']])
print(b['abbr'].value_counts())

  name abbr   wOBA
8  張育成    悍  0.410
6   魔鷹    鷹  0.404
0  陳傑憲    獅  0.388
3  潘傑楷    獅  0.381
4  陳晨威    猿  0.367
   name abbr    FIP
52  蘇俊璋    猿  1.811
40  曾峻岳    悍  1.812
63  威能帝    猿  1.926
59  呂彥青    象  2.140
0    蔣銲    龍  2.196
abbr
悍    34
猿    32
鷹    31
象    31
獅    30
龍    25
Name: count, dtype: int64
